# Construindo um Cubo Resumo de Garantia de Receita de Telecom com PROC SUMMARY

## Resumo Executivo

Uma equipe de garantia de receita de uma operadora sem fio pré-agrega um mês de registros de faturamento diário por assinante em um cubo resumo compacto, para que os analistas possam detalhar a receita liquidada por região, nível de plano e tipo de chamada sem reescanear a tabela de fatos bruta. O `PROC SUMMARY` consolida 100 registros diários de assinante em um conjunto multidimensional de células: a granularidade mais fina (região x nível de plano x tipo de chamada) preenche 25 das 27 células possíveis, enquanto subcubos nomeados fornecem os totais marginais que os analistas mais consultam. Neste mês de amostra, a operadora liquidou \$222.78 nas três regiões ativas, com Sul (\$97.44) e Leste (\$86.94) juntos respondendo por 83% da receita, e Norte na retaguarda com \$38.40. O subcubo individual mais rico é o serviço de Voz no nível Plus (\$59.06 em 18 registros), e classificar as células pelo rendimento por registro revela as células de uso de dados como os alvos de maior valor para uma auditoria de vazamento (rendimento máximo \$7.87/registro). Cada número abaixo é lido diretamente da saída executada.

## Fontes de Dados

| Conjunto de Dados | Linhas | Grão | Variáveis-Chave |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Uma linha por resumo diário de uso do assinante | `region` (East/South/North), `plan_tier` (Prepaid/Basic/Plus), `call_type` (Voice/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Uma linha por célula não vazia (region x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Uma linha por célula dos subcubos de detalhamento nomeados | `_TYPE_`, `_FREQ_`, `rev_total` |

Todos os dados são gerados inline com `call streaminit()` / `rand()`; não são necessários arquivos externos nem acesso à rede. Este ambiente é executado sem licença, então os conjuntos de dados gravados são limitados a 100 observações — o cenário é dimensionado para que o cubo fique totalmente populado dentro desse limite.

## De registros brutos de detalhe de chamadas a um cubo navegável

As operadoras sem fio liquidam receita em milhões de eventos diários de uso. Para permitir que analistas de garantia de receita respondam a perguntas como *"Qual foi a receita de voz do nível Plus na região Sul no mês passado?"* sem reescanear a tabela de fatos bruta toda vez, nós **pré-agregamos** os dados em um cubo resumo compacto.

O `PROC SUMMARY` é o cavalo de batalha do Base SAS exatamente para isso: ele agrupa uma tabela de fatos plana por uma ou mais dimensões `CLASS` e grava as estatísticas solicitadas em um conjunto de dados de saída, marcando cada linha com `_TYPE_` (quais dimensões estão ativas) e `_FREQ_` (registros por trás da célula). Esse conjunto de dados de saída *é* um cubo multidimensional — a mesma estrutura de consolidação que uma ferramenta OLAP exporia, materializada como um conjunto de dados SAS comum que você pode imprimir, unir e fatiar.

Este notebook:

1. Gera um mês realista de registros de faturamento diário por assinante.
2. Constrói o cubo de granularidade mais fina (region x plan_tier x call_type) com `PROC SUMMARY NWAY`.
3. Materializa **subcubos de detalhamento** nomeados com a instrução `TYPES`.
4. Projeta a receita para a base de assinantes com um `WEIGHT`, detalha em uma região e classifica as células por rendimento por registro para triagem de vazamento.

## Etapa 1 - Gerar dados sintéticos de detalhe de chamadas / faturamento

Cada linha resume o uso de um assinante de um serviço em um dia. Usamos `call streaminit` para reprodutibilidade e `rand()` para sortear distribuições plausíveis: receita e uso escalam com o nível do plano, a receita de voz acompanha os minutos faturáveis, e a receita de dados acompanha os megabytes. Cada `RAND('table', ...)` recebe uma probabilidade por categoria para que cada região, nível e tipo de chamada apareça na amostra de 100 registros. Um pequeno peso de pesquisa `subscriber_wt` é anexado para que possamos demonstrar uma medida ponderada depois.

In [1]:
DADOS work.cdr_billing;
    CHAMAR streaminit(20260131);
    COMPRIMENTO region $6 plan_tier $14 call_type $7 device_class $16 bill_month $7;
    RÓTULO revenue       = "Receita Liquidada (USD)"
          call_minutes  = "Minutos de Voz Faturáveis"
          data_mb       = "Volume de Dados (MB)"
          subscriber_wt = "Peso de Pesquisa do Assinante"
          region        = "Região"
          plan_tier     = "Nível do Plano"
          call_type     = "Tipo de Chamada"
          device_class  = "Tipo de Aparelho"
          bill_month    = "Mês de Faturamento";

    FAZER i = 1 ATÉ 100;
        /* --- Dimensões: uma probabilidade por nível, somando 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        SELECIONAR (r);
            QUANDO (1) region = "Leste";
            QUANDO (2) region = "Sul";
            OUTROS region = "Norte";
        FIM;

        p = rand("table", 0.30, 0.40, 0.30);
        SELECIONAR (p);
            QUANDO (1) plan_tier = "Pré-pago";
            QUANDO (2) plan_tier = "Básico";
            OUTROS plan_tier = "Plus";
        FIM;

        c = rand("table", 0.50, 0.30, 0.20);
        SELECIONAR (c);
            QUANDO (1) call_type = "Voz";
            QUANDO (2) call_type = "SMS";
            OUTROS call_type = "Dados";
        FIM;

        SE rand("uniform") < 0.55 ENTÃO device_class = "Smartphone";
        SENÃO device_class = "Convencional";

        bill_month = "2026-01";

        /* --- Medidas, escaladas por nível e serviço --- */
        tier_mult = (plan_tier = "Pré-pago")*0.7
                  + (plan_tier = "Básico")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Voz")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Dados")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        SAÍDA;
    FIM;
    REMOVER i r p c tier_mult base_rev;
EXECUTAR;

PROC PRINT DADOS=work.cdr_billing(obs=8) RÓTULO noobs;
    TÍTULO "Amostra de Registros de Faturamento Diário por Assinante";
EXECUTAR;

                                Amostra de Registros de Faturamento Diário por Assinante                                

 Região   Nível do Plano  Tipo de Chamada  Tipo de Aparelho   Mês de Faturamento   Minutos de Voz Faturáveis  Volume de Dados (MB)  Receita Liquidada (USD)  Peso de Pesquisa do Assinante
Norte    Plus             SMS              Smartphone        2026-01                                       0                     0                     0.67                           1.13
Sul      Pré-pago         SMS              Convencional      2026-01                                       0                     0                     0.94                           1.42
Sul      Plus             SMS              Smartphone        2026-01                                       0                     0                     0.99                           0.86
Sul      Plus             SMS              Smartphone        2026-01                                       0                     0


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Etapa 2 - Construir o cubo de granularidade mais fina com PROC SUMMARY NWAY

`NWAY` mantém apenas a combinação mais detalhada de todas as dimensões `CLASS` - aqui, cada célula populada (region x plan_tier x call_type). Para cada célula armazenamos a `SUM`, `MEAN` e `MAX` da receita, mais o total de minutos e megabytes, para que um analista possa ler a receita total por célula, derivar a média por registro e identificar a maior cobrança individual. `_FREQ_` registra quantos dias-assinante estão por trás de cada célula. Descartamos `_TYPE_` aqui porque, na granularidade NWAY, toda linha tem o mesmo tipo.

In [2]:
PROC summary DADOS=work.cdr_billing NWAY;
    CLASSE region plan_tier call_type;
    VARIÁVEL revenue call_minutes data_mb;
    SAÍDA out=work.cube_nway(drop=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
EXECUTAR;

PROC PRINT DADOS=work.cube_nway(obs=12) RÓTULO noobs;
    TÍTULO "Células do Cubo NWAY (region * plan_tier * call_type)";
    RÓTULO region="Região" plan_tier="Nível do Plano" call_type="Tipo de Chamada"
          rev_total="Receita Total" rev_mean="Receita Média" rev_max="Receita Máxima"
          min_total="Minutos Totais" data_total="Dados Totais (MB)";
    FORMATO rev_total rev_mean rev_max min_total data_total comma10.2;
EXECUTAR;

                                 Células do Cubo NWAY (region * plan_tier * call_type)                                  

 Região   Nível do Plano  Tipo de Chamada  _FREQ_  Receita Total   Receita Média   Receita Máxima  Minutos Totais  Dados Totais (MB)
Leste    Básico           Dados                 4          18.17            4.54             8.05            0.00           1,807.90
Leste    Básico           SMS                   4           4.07            1.02             1.24            0.00               0.00
Leste    Básico           Voz                   7          15.08            2.15             3.73          302.50               0.00
Leste    Plus             Dados                 1           5.54            5.54             5.54            0.00             573.90
Leste    Plus             SMS                   4           3.59            0.90             0.95            0.00               0.00
Leste    Plus             Voz                   7          23.87            3.41


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Etapa 3 - Materializar subcubos de detalhamento nomeados com TYPES

Um cubo OLAP pré-armazena as consolidações que os analistas mais navegam. A instrução `TYPES` faz exatamente isso: cada termo pede ao `PROC SUMMARY` para emitir um subcubo. Solicitamos o total geral `()`, o marginal de `region`, e os subcubos bidimensionais `region*plan_tier` e `call_type*plan_tier` - os caminhos de detalhamento que um painel de receita exporia. O SAS marca cada subcubo com um código `_TYPE_` (uma máscara de bits sobre a lista `CLASS`), de modo que um único conjunto de dados de saída carrega todo nível da hierarquia.

In [3]:
PROC summary DADOS=work.cdr_billing;
    CLASSE region plan_tier call_type;
    VARIÁVEL revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    SAÍDA out=work.cube_hier
           sum(revenue)=rev_total;
EXECUTAR;

PROC PRINT DADOS=work.cube_hier RÓTULO noobs;
    TÍTULO "Consolidações de Subcubo: total geral, region, region*tier, calltype*tier";
    VARIÁVEL _type_ region plan_tier call_type _freq_ rev_total;
    RÓTULO region="Região" plan_tier="Nível do Plano" call_type="Tipo de Chamada"
          rev_total="Receita Total";
    FORMATO rev_total comma10.2;
EXECUTAR;

                       Consolidações de Subcubo: total geral, region, region*tier, calltype*tier                        

_TYPE_   Região   Nível do Plano  Tipo de Chamada  _FREQ_  Receita Total
     0                                                100         222.78
     3           Básico           Dados                 8          38.06
     3           Básico           SMS                   8           8.03
     3           Básico           Voz                  20          42.33
     3           Plus             Dados                 3          17.46
     3           Plus             SMS                  13          11.75
     3           Plus             Voz                  18          59.06
     3           Pré-pago         Dados                 7          14.50
     3           Pré-pago         SMS                   7           6.82
     3           Pré-pago         Voz                  16          24.77
     4  Leste                                          38          86.94
  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Etapa 4 - Projeção ponderada, um detalhamento regional e triagem de vazamento

Três leituras que uma equipe de garantia de receita realmente executa contra o cubo:

- **Projeção ponderada.** Anexar `WEIGHT=subscriber_wt` a um resumo `region*plan_tier` relata a receita escalada para toda a base de assinantes que a amostra representa, em vez das linhas brutas amostradas.
- **Detalhamento.** Filtrar o cubo NWAY para uma região expande um único ramo da hierarquia - aqui Sul - para seu detalhe de nível por serviço.
- **Triagem de vazamento.** Ordenar as células pela receita média por registro revela as células de maior rendimento; células de frequência baixa e alto rendimento são exatamente o que uma auditoria examina em busca de receita mal tarifada ou vazada.

In [4]:
/* Receita ponderada projetada para a base de assinantes */
PROC summary DADOS=work.cdr_billing NWAY;
    CLASSE region plan_tier;
    VARIÁVEL revenue;
    PESO subscriber_wt;
    SAÍDA out=work.cube_wt(drop=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
EXECUTAR;

PROC PRINT DADOS=work.cube_wt RÓTULO noobs;
    TÍTULO "Receita ponderada por region * plan_tier (projetada para a base de assinantes)";
    RÓTULO region="Região" plan_tier="Nível do Plano"
          rev_weighted="Receita Ponderada" records="Registros";
    FORMATO rev_weighted comma10.2;
EXECUTAR;

/* Detalhar no ramo da região Sul do cubo */
PROC PRINT DADOS=work.cube_nway RÓTULO noobs;
    ONDE region = "Sul";
    TÍTULO "Detalhamento de Sul: células de receita por nível e tipo de chamada";
    VARIÁVEL plan_tier call_type _freq_ rev_total rev_mean;
    RÓTULO plan_tier="Nível do Plano" call_type="Tipo de Chamada"
          rev_total="Receita Total" rev_mean="Receita Média";
    FORMATO rev_total rev_mean comma10.2;
EXECUTAR;

/* Classificar células por rendimento por registro para triagem de vazamento */
PROC SORT DADOS=work.cube_nway out=work.cube_ranked;
    POR DECRESCENTE rev_mean;
EXECUTAR;

PROC PRINT DADOS=work.cube_ranked(obs=6) RÓTULO noobs;
    TÍTULO "Células de maior receita média (rendimento por registro)";
    VARIÁVEL region plan_tier call_type _freq_ rev_mean rev_max;
    RÓTULO region="Região" plan_tier="Nível do Plano" call_type="Tipo de Chamada"
          rev_mean="Receita Média" rev_max="Receita Máxima";
    FORMATO rev_mean rev_max comma10.2;
EXECUTAR;

                     Receita ponderada por region * plan_tier (projetada para a base de assinantes)                     

 Região   Nível do Plano  Receita Ponderada  Registros
Leste    Básico                       50.85         15
Leste    Plus                         59.59         12
Leste    Pré-pago                     29.77         11
Norte    Básico                       18.30          7
Norte    Plus                         22.42          7
Norte    Pré-pago                     20.56          9
Sul      Básico                       58.63         14
Sul      Plus                         56.29         15
Sul      Pré-pago                     27.77         10

                          Detalhamento de Sul: células de receita por nível e tipo de chamada                           

 Nível do Plano  Tipo de Chamada  _FREQ_  Receita Total   Receita Média
Básico           Dados                 3          12.02            4.01
Básico           SMS                   2           2.01      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpretando os resultados

O cubo resumo transforma 100 linhas brutas de dia-assinante em um conjunto compacto de células pré-agregadas que respondem instantaneamente a perguntas de detalhamento, sem reescanear a tabela de fatos:

- **Onde a receita está.** O marginal de `region` (`_TYPE_=4`) mostra que Sul liquidou \$97.44 e Leste \$86.94 - juntos 83% dos \$222.78 do mês - enquanto Norte contribuiu com \$38.40. Dentro do subcubo `call_type*plan_tier` (`_TYPE_=3`), Voz no nível Plus é a célula individual mais rica, com \$59.06 em 18 registros, seguida por Voz no nível Basic, com \$42.33.
- **Projeção ponderada.** Aplicar o peso de pesquisa reembaralha a classificação em direção aos planos cujos assinantes carregam mais peso: Leste-Plus (\$59.59) e Sul-Basic (\$58.63) lideram a receita projetada de `region*plan_tier`, um quadro diferente dos totais regionais não ponderados e um lembrete para relatar receita projetada, e não amostrada, ao dimensionar exposição.
- **Rendimento por registro e triagem de vazamento.** Classificar as células NWAY por `rev_mean` coloca as células de uso de dados no topo - Norte-Basic-Dados (\$7.87/registro) e Sul-Plus-Dados (\$5.96/registro) - confirmando que o uso intenso de dados gera a maior receita por registro. Os líderes de frequência baixa (um ou dois registros) são precisamente as células para as quais um analista de garantia de receita puxaria os registros de detalhe de chamada subjacentes, para confirmar que a cobrança alta está corretamente tarifada, e não é um erro.

Para uma equipe de garantia de receita, esse cubo é a base para painéis de variância: comparar a receita liquidada com a receita esperada de tabela de tarifas por célula (região, nível, tipo de chamada), e as células cujo total médio ou ponderado mais se desviam se tornam os casos de vazamento que vale a pena auditar. Como toda a estrutura é um conjunto de dados SAS comum, o cubo do próximo mês pode ser unido, comparado ou combinado com uma tabela de tarifas com as mesmas ferramentas do Base SAS.